In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import scipy.io as io
import argparse
from tqdm import tqdm

import src.utils.utils as utils

if torch.cuda.is_available():
    dev = "cuda:0"
    torch.set_default_device(dev)
else:
    print(f"{torch.cuda.is_available()}")
    dev = "cpu"

dataset = "urban"
data = io.loadmat(f"datasets/{dataset}.mat")
Y_flat = torch.tensor(data["Y"], dtype=torch.float32)
Y_flat = utils.normalize(Y_flat, dim=0)
E = torch.tensor(data["E"])
B, c, N = E.shape[0], E.shape[1], Y_flat.shape[1]

/home/ids/edabier/miniconda3/envs/hsu-env/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/ids/edabier/miniconda3/envs/hsu-env/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may hav

False
False


In [2]:
import sys
sys.path.append("/home/ids/edabier/HSU/HyperSIGMA")
from HyperSIGMA.HyperspectralUnmixing.models.SpatVit import Block
from functools import partial
import torch.utils.checkpoint as checkpoint
import torch.nn.functional as F

img_size, embed_dim, patch_size, num_heads, depth, mlp_ratio, qkv_bias, qk_scale, drop_rate, attn_drop_rate = 64, 768, 2, 12, 12, 4., True, None, 0., 0.
drop_path_rate, init_values, interval, restart_regression, n_points, out_indices, NUM_TOKENS = 0.1, None, 3, True, 8, [3,5,7,11], 64
norm_layer = partial(nn.LayerNorm, eps=1e-6)
dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]

def get_reference_points(spatial_shapes, device):
    # for lvl, (H_, W_) in enumerate(spatial_shapes):
    H_, W_ = spatial_shapes[0], spatial_shapes[1]
    ref_y, ref_x = torch.meshgrid(
        torch.linspace(0.5, H_ - 0.5, H_, dtype=torch.float32, device=device),
        torch.linspace(0.5, W_ - 0.5, W_, dtype=torch.float32, device=device))
    ref_y = ref_y.reshape(-1)[None] / H_
    ref_x = ref_x.reshape(-1)[None] / W_
    ref = torch.stack((ref_x, ref_y), -1)  # 每个点归一化后的坐标，尺寸(1，H_*W_，2)
    return ref

def deform_PatchInputs_func(x, seg_patch_size):
    B, c, h, w = x.shape
    b = B // 3
    spatial_shapes = torch.as_tensor([h // seg_patch_size, w // seg_patch_size],
                                     dtype=torch.long, device=x.device)
    reference_points = get_reference_points([h // seg_patch_size, w // seg_patch_size],
                                            x.device)  # [h // 8, w // 8], x.device)
    deform_inputs = [reference_points, spatial_shapes]
    return deform_inputs

class PatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """
    def __init__(self, img_size=224, patch_size=64, in_chans=3, embed_dim=768):
        super().__init__()

        img_size = (img_size, img_size)
        patch_size = (patch_size, patch_size)
        num_patches = (img_size[1] // patch_size[1]) * (img_size[0] // patch_size[0])
        self.patch_shape = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = num_patches

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x, **kwargs):
        B, C, H, W = x.shape
        x = self.proj(x)
        Hp, Wp = x.shape[2], x.shape[3]

        x = x.flatten(2).transpose(1, 2)
        return x, (Hp, Wp)

pos_drop = nn.Dropout(p=drop_rate)
conv_features = nn.Conv2d(embed_dim, NUM_TOKENS, kernel_size=1, bias=False)

conv1 =nn.Sequential(
    nn.Conv2d(NUM_TOKENS, NUM_TOKENS, kernel_size=1),
    nn.LeakyReLU(0.02),
    nn.BatchNorm2d(NUM_TOKENS),
    nn.Dropout(0.2),
    )
conv2 = nn.Sequential(
    nn.Conv2d(NUM_TOKENS, NUM_TOKENS, kernel_size=3, padding=1),
    nn.LeakyReLU(0.02),
    nn.BatchNorm2d(NUM_TOKENS),
    nn.Dropout(0.2),
)
smooth = nn.Conv2d(NUM_TOKENS*4, NUM_TOKENS, kernel_size=3, stride=1, padding=1)
conv3 = nn.Sequential(
    nn.Conv2d(NUM_TOKENS, c, kernel_size=1),
    nn.LeakyReLU(0.02),
    nn.BatchNorm2d(c),
    nn.Dropout(0.2),
)

blocks = nn.ModuleList([
            Block(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
                init_values=init_values, sample=((i + 1) % interval != 0),
                restart_regression=restart_regression, n_points=n_points)
            for i in range(depth)])

patch_embed = PatchEmbed(img_size=img_size, patch_size=2, in_chans=B)

x = torch.rand(1, B, img_size, img_size)
img = [x]
deform_inputs = deform_PatchInputs_func(x, 2)
x, (Hp, Wp) = patch_embed(x)
x = pos_drop(x)

features = []
for i, blk in enumerate(blocks):
    x = checkpoint.checkpoint(blk, x, Hp, Wp, deform_inputs)

    if i in out_indices:
        features.append(x)
features = list(map(lambda x: x.permute(0, 2, 1).reshape(1, -1, Hp, Wp), features))

ops = [nn.Identity(), nn.Identity(), nn.Identity(), nn.Identity()]
for i in range(len(ops)):
    features[i] = ops[i](features[i])

Feat_spat = img + features
x = []
x.append(Feat_spat[0])
ops = [conv_features, conv_features, conv_features, conv_features]
for i in range(len(ops)):
    x.append(ops[i](Feat_spat[i+1]))
img_fea = x[1:]

p4 = conv1(x[4])
p3 = conv1(x[3])
p2 = conv2(x[2])
p1 = conv2(x[1])
p1 = torch.cat([p1,p2,p3,p4], dim=1)

p1 = F.interpolate(p1, size=(64, 64), mode='bilinear', align_corners=True)
p1 = smooth(p1)
x = conv3(p1)
x.shape

/home/ids/edabier/miniconda3/envs/hsu-env/lib/python3.10/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/home/ids/edabier/miniconda3/envs/hsu-env/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


torch.Size([1, 6, 64, 64])

In [3]:
from HyperSIGMA.HyperspectralUnmixing.models.SpecVit import SpecViT
import math

spec_vit = SpecViT(NUM_TOKENS=NUM_TOKENS, img_size=patch_size, in_chans=B,drop_path_rate=0.1, out_indices=[3, 5, 7, 11],
            embed_dim=768, depth=12, num_heads=12, mlp_ratio=4, qkv_bias=True, qk_scale=None,
            drop_rate=0., attn_drop_rate=0., use_checkpoint=False, use_abs_pos_emb=False, interval=3)

def get_reference_points(spatial_shapes, device):
    H_, W_ = spatial_shapes[0], spatial_shapes[1]
    ref_y, ref_x = torch.meshgrid(
        torch.linspace(0.5, H_ - 0.5, H_, dtype=torch.float32, device=device),
        torch.linspace(0.5, W_ - 0.5, W_, dtype=torch.float32, device=device))
    ref_y = ref_y.reshape(-1)[None] / H_
    ref_x = ref_x.reshape(-1)[None] / W_
    ref = torch.stack((ref_x, ref_y), -1)  # 每个点归一化后的坐标，尺寸(1，H_*W_，2)
    return ref

def deform_inputs_func(x, num_tokens):
    spatial_shapes = torch.as_tensor([int(math.sqrt(num_tokens)), int(math.sqrt(num_tokens))],
                                     dtype=torch.long, device=x.device)  # 3*2的tensor
    reference_points = get_reference_points([int(math.sqrt(num_tokens)), int(math.sqrt(num_tokens))], x.device)
    deform_inputs = [reference_points, spatial_shapes]

    return deform_inputs

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224):
        super().__init__()

        img_size = (img_size,img_size)
        num_patches = (img_size[1]) * (img_size[0])
        patch_shape = (img_size[0], img_size[1])

    def forward(self, x, **kwargs):
        x = x.flatten(2).transpose(1, 2)
        return x

norm_layer = partial(nn.LayerNorm, eps=1e-6)
patch_embed = PatchEmbed(img_size=img_size)

spec_embed = nn.AdaptiveAvgPool1d(NUM_TOKENS)
spat_map = nn.Linear(int(img_size * img_size), embed_dim)

pos_drop = nn.Dropout(p=drop_rate)

dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]  # stochastic depth decay rule

blocks = nn.ModuleList([
    Block(
        dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
        drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[i], norm_layer=norm_layer,
        init_values=init_values, sample=((i + 1) % interval != 0),
        restart_regression=restart_regression, n_points=n_points)
    for i in range(depth)])

norm = norm_layer(embed_dim)
l = nn.Linear(embed_dim, 128, bias=False)

x = torch.rand(1, B, img_size, img_size)

deform_inputs = deform_inputs_func(x, NUM_TOKENS)
x = patch_embed(x)
print(x.shape)
x = spec_embed(x)
print(x.shape)
_, _, num_tokens = x.shape
x = x.transpose(1, 2)
x_in = x.reshape(1, num_tokens, img_size, img_size)
x = spat_map(x)
print(x.shape)
batch_size, _, embed_dim = x.size()
x = pos_drop(x)

features = []
for i, blk in enumerate(blocks):
    x = blk(x, img_size, img_size, deform_inputs)

    if i in out_indices:
        features.append(x)  # b, channels, embed_dim

ops = [l, l, l, l]
for i in range(len(ops)):
    features[i] = ops[i](features[i])

Feat_spec = features

torch.Size([1, 4096, 162])
torch.Size([1, 4096, 64])
torch.Size([1, 64, 768])


In [4]:
pool = nn.AdaptiveAvgPool1d(1)
fc_spec = nn.Sequential(
            nn.Linear(NUM_TOKENS, 32, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(32, NUM_TOKENS, bias=False),
            nn.Sigmoid(),
        )

spec_feature = Feat_spec[-1]
spec_feature = pool(spec_feature).view(1, -1)

spec_weights = []
ops_ = [fc_spec, fc_spec, fc_spec, fc_spec]
for i in range(len(ops_)):
    spec_weights.append((ops_[i](spec_feature)).view(1, -1, 1, 1))
print(spec_weights[0].shape)

ss_feature = []
ss_feature.append(x)
for i in range(4):
    ss_feature.append((1 + spec_weights[i]) * img_fea[i])

x = ss_feature
p4 = conv1(x[4])
p3 = conv1(x[3])
p2 = conv2(x[2])
p1 = conv2(x[1])
p1 = torch.cat([p1,p2,p3,p4], dim=1)

p1 = F.interpolate(p1, size=(64, 64), mode='bilinear', align_corners=True)
p1 = smooth(p1)
x = conv3(p1)
x.shape

torch.Size([1, 64, 1, 1])


torch.Size([1, 6, 64, 64])